# Expected VE for a planned run

Takes a run plan, a personal profile and optional running history, and returns the expected pulmonary ventilation
(VE, litres of air per minute) with an error band. HazeWise multiplies VE by duration and PM2.5 to get the inhaled
dose.

The model is the log-VE regression from `explore.ipynb` §12, refitted on the 992 Málaga treadmill tests. Which
version runs depends on what history the user gives:

| history given | variant | inputs |
|---|---|---|
| none | A | speed, age, sex, weight, height |
| a recent best effort (race or all-out run) | B | A + peak running speed estimated from that effort |
| recent runs with average HR, no best effort | C | A + HR at the goal pace |

Sections 4 and 5 test all of these, plus a variant D that uses both HR and peak speed. D never beats B on history
data, so the final model doesn't use it.

## 1. Inputs

In [1]:
PLAN = {"pace": "6:00", "distance_km": 10.0}            # goal pace, min:sec per km
PERSON = {"age": 30, "sex": "male", "height_cm": 175, "weight_kg": 70}
HISTORY = {                                             # optional: set either part to None / []
    "best_effort": {"distance_km": 5.0, "time": "25:00"},          # a recent race or all-out effort
    "runs": [                                                     # recent steady runs with average HR
        {"distance_km": 8.0, "time": "52:00", "avg_hr": 148},
        {"distance_km": 5.0, "time": "30:00", "avg_hr": 155},
        {"distance_km": 10.0, "time": "58:00", "avg_hr": 160},
    ],
}

## 2. Data

Same cleaning and 10 s medians as `explore.ipynb` §1 and §2. The modelling set is every exercise bin (ramp, plus
steady walking after the first minute of warm-up) with speed, HR and profile present. Unlike `explore.ipynb` §12 it
keeps the 11 tests without gas data, since nothing here needs VO2.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold

DATA = Path("data/treadmill-maximal-exercise-tests-from-the-exercise-physiology-and-human-performance-lab-of-the-university-of-malaga-1.0.1")
pd.set_option("display.precision", 2)
pd.set_option("display.width", 140)

meas = pd.read_csv(DATA / "test_measure.csv")
subj = pd.read_csv(DATA / "subject-info.csv")

m = meas.sort_values(["ID_test", "time"]).reset_index(drop=True)
m.loc[m.HR < 40, "HR"] = np.nan
g = m.groupby("ID_test")
hr_med = g.HR.transform(lambda s: s.rolling(15, center=True, min_periods=5).median())
m.loc[(m.HR - hr_med).abs() > 20, "HR"] = np.nan
ve_med = g.VE.transform(lambda s: s.rolling(7, center=True, min_periods=3).median())
m = m[~((m.VE - ve_med).abs() > 0.5 * ve_med)].copy()

g = m.groupby("ID_test")
pk = g.Speed.transform("max")
recovery = (m.Speed == pk).groupby(m.ID_test).cummax() & (m.Speed < pk)
moving = m.Speed > 0
t_move0 = m.time.where(moving).groupby(m.ID_test).transform("min")
warm_speed = m.Speed.where(moving & (m.time < t_move0 + 60)).groupby(m.ID_test).transform("median")
t_ramp0 = m.time.where(~recovery & (m.Speed <= warm_speed + 0.05)).groupby(m.ID_test).transform("max")
m["phase"] = np.select([recovery, ~recovery & (m.time > t_ramp0), ~recovery & moving, ~recovery & ~moving],
                       ["recovery", "ramp", "warmup", "rest"], default="other")
m["t_phase"] = m.time - m.groupby(["ID_test", "phase"]).time.transform("min")

m["tbin"] = (m.time // 10).astype(int)
bins = (m.groupby(["ID_test", "ID", "phase", "tbin"])[["time", "t_phase", "Speed", "HR", "VE"]].median().reset_index()
        .merge(subj.drop(columns="ID"), on="ID_test", how="left"))
ex = bins[(bins.phase == "ramp") | ((bins.phase == "warmup") & (bins.t_phase >= 60))]
d = ex.dropna(subset=["Speed", "HR", "VE", "Age", "Sex", "Weight", "Height"]).reset_index(drop=True)
vpeak_of = m.groupby("ID_test").Speed.max()
d["vpeak"] = d.ID_test.map(vpeak_of)
y = d.VE.to_numpy()
print(f"{len(d):,} bins from {d.ID_test.nunique()} tests / {d.ID.nunique()} subjects")

82,274 bins from 992 tests / 857 subjects


## 3. Model

Ordinary least squares on log VE. Speed enters as a linear term plus hinges `max(0, speed − k)` at 6.5, 8, 10, 13
and 16 km/h, so the slope can change at each knot. Weight enters as log weight, the rest linearly. The optional
blocks:

* HR (C, D): the same hinge treatment at 100 to 180 bpm, plus a speed × HR term. At a given speed, a higher HR
  means a less fit person, and less fit people breathe more.
* peak speed (B, D): `vpeak` and `vpeak × speed`. That's the top speed of the test, standing in for the fitness a
  best effort reveals.

Predictions go back to litres with Duan smearing. `fit()` can also train on copies of the data with a random
per-test offset added to HR or vpeak. That teaches the fit to trust an input only as much as a history-derived
version of it deserves. Section 5 sets the offset sizes from repeat tests.

In [3]:
SPEED_KNOTS = [6.5, 8, 10, 13, 16]
HR_KNOTS = [100, 120, 140, 160, 180]
VARIANTS = {"A": dict(hr=False, vpeak=False), "B": dict(hr=False, vpeak=True),
            "C": dict(hr=True, vpeak=False), "D": dict(hr=True, vpeak=True)}


def design(df, hr, vpeak):
    X = pd.DataFrame({"Speed": df.Speed}, index=df.index)
    for k in SPEED_KNOTS:
        X[f"max(0,Speed-{k})"] = (df.Speed - k).clip(lower=0)
    X["Age"], X["Sex"], X["logW"], X["Height"] = df.Age, df.Sex, np.log(df.Weight), df.Height
    if hr:
        X["HR"] = df.HR
        for k in HR_KNOTS:
            X[f"max(0,HR-{k})"] = (df.HR - k).clip(lower=0)
        X["Speed*HR"] = df.Speed * df.HR
    if vpeak:
        X["vpeak"] = df.vpeak
        X["vpeak*Speed"] = df.vpeak * df.Speed
    return X


class Formula:
    def __init__(self, df, hr=False, vpeak=False, hr_noise=0.0, vpeak_noise=0.0, copies=3, seed=0):
        if (hr and hr_noise) or (vpeak and vpeak_noise):
            rng = np.random.default_rng(seed)
            ids = df.ID_test.unique()
            df = pd.concat([df.assign(
                HR=df.HR + df.ID_test.map(pd.Series(rng.normal(0, hr_noise, len(ids)), index=ids)),
                vpeak=df.vpeak + df.ID_test.map(pd.Series(rng.normal(0, vpeak_noise, len(ids)), index=ids)))
                for _ in range(copies)])
        self.hr, self.vpeak = hr, vpeak
        X, t = design(df, hr, vpeak), np.log(df.VE)
        self.ols = LinearRegression().fit(X, t)
        self.smear = np.mean(np.exp(t - self.ols.predict(X)))

    def predict(self, df):
        return np.exp(self.ols.predict(design(df, self.hr, self.vpeak))) * self.smear

    def coefs(self):
        return pd.Series(np.r_[self.ols.intercept_, self.ols.coef_, self.smear],
                         index=["intercept", *self.ols.feature_names_in_, "smear"])


def scores(y, p):
    ape = np.abs(p - y) / y
    return {"MAE L/min": np.mean(np.abs(p - y)), "median APE %": 100 * np.median(ape),
            "90th pct APE %": 100 * np.quantile(ape, 0.9), "R²": 1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2)}


folds = list(GroupKFold(n_splits=5).split(d, groups=d.ID))

## 4. Test: same-session inputs, all subjects

Cross-validated by subject: 5 folds, no person in both train and test. HR and peak speed come from the same test
the model predicts. That's the best case, since an app only ever has them from earlier runs.

In [4]:
oof = {k: np.empty(len(d)) for k in VARIANTS}
for tr, te in folds:
    for k, v in VARIANTS.items():
        oof[k][te] = Formula(d.iloc[tr], **v).predict(d.iloc[te])
display(pd.DataFrame({k: scores(y, p) for k, p in oof.items()}).T)

,MAE L/min,median APE %,90th pct APE %,R²
A,8.68,11.81,31.01,0.85
B,7.62,10.42,28.02,0.89
C,8.12,10.95,28.57,0.87
D,7.58,10.34,27.29,0.89


Peak speed is worth more than HR. It takes median error from 11.8% to 10.4%, and HR only gets it to 11.0%. Both
together (D) reach 10.3%, barely ahead of peak speed alone. These are best cases, since an app never has same-day
values.

## 5. Test: history from a different session

116 people did two or more tests. For each ordered pair of tests by the same person, the earlier or later test plays
"history" and the model predicts VE in the other:

* HR at pace: the history test's median HR at each speed, read off at the target bin's speed. Target bins outside
  the history test's speed range are skipped.
* peak speed: the history test's peak.

This includes day-to-day HR variation and real fitness change between tests. It still uses lab ramp HR on both
sides. A watch reading from an outdoor run will be noisier.

First, how far apart history and target are. Those spreads set the noise sizes used for training.

In [5]:
curves = {t: g_.groupby(g_.Speed.round(1)).HR.median() for t, g_ in d.groupby("ID_test")}
tests_of = d.groupby("ID").ID_test.unique()
repeat = tests_of[tests_of.map(len) > 1]
pairs = [(h, t) for ts in repeat for h in ts for t in ts if h != t]
by_test = dict(tuple(d.groupby("ID_test")))


def with_history(target, hist_id):
    c = curves[hist_id]
    t = target[target.Speed.between(c.index.min(), c.index.max())]
    return t.assign(HR=np.interp(t.Speed, c.index, c.values), vpeak=vpeak_of[hist_id])


hist_err = pd.DataFrame([{
    "hr_off": np.median(with_history(by_test[t], h).HR - by_test[t].HR.loc[with_history(by_test[t], h).index]),
    "vpeak_off": vpeak_of[h] - vpeak_of[t],
    "age_gap": abs(by_test[h].Age.iloc[0] - by_test[t].Age.iloc[0]),
} for h, t in pairs])
robust_sd = lambda s: (s.quantile(0.75) - s.quantile(0.25)) / 1.349
HR_NOISE_SD, VPEAK_NOISE_SD = robust_sd(hist_err.hr_off), robust_sd(hist_err.vpeak_off)
print(f"{len(repeat)} subjects, {len(pairs)} ordered pairs; years between tests: median {hist_err.age_gap.median():.1f}, "
      f"IQR {hist_err.age_gap.quantile(.25):.1f}-{hist_err.age_gap.quantile(.75):.1f}")
print(f"history HR minus actual HR at the same speed: median {hist_err.hr_off.median():+.1f} bpm, robust SD {HR_NOISE_SD:.1f}")
print(f"history peak speed minus actual: median {hist_err.vpeak_off.median():+.2f} km/h, robust SD {VPEAK_NOISE_SD:.2f}")

116 subjects, 316 ordered pairs; years between tests: median 0.7, IQR 0.3-1.3
history HR minus actual HR at the same speed: median +0.0 bpm, robust SD 6.9
history peak speed minus actual: median +0.00 km/h, robust SD 0.89


In [6]:
CANDIDATES = {
    "A": dict(hr=False, vpeak=False),
    "B": dict(hr=False, vpeak=True),
    "B, noise-trained": dict(hr=False, vpeak=True, vpeak_noise=VPEAK_NOISE_SD),
    "C": dict(hr=True, vpeak=False),
    "C, noise-trained": dict(hr=True, vpeak=False, hr_noise=HR_NOISE_SD),
    "D": dict(hr=True, vpeak=True),
    "D, noise-trained": dict(hr=True, vpeak=True, hr_noise=HR_NOISE_SD, vpeak_noise=VPEAK_NOISE_SD),
}
rows = []
for tr, te in folds:
    fits = {k: Formula(d.iloc[tr], **v) for k, v in CANDIDATES.items()}
    test_ids = set(d.ID_test.iloc[te])
    for h, t in pairs:
        if t not in test_ids:
            continue
        hist = with_history(by_test[t], h)
        if len(hist) < 10:
            continue
        rows.append(pd.DataFrame({"ID": by_test[t].ID.iloc[0], "y": hist.VE.to_numpy(), "ID_test": t, "pair": f"{h}>{t}",
                                  **{k: f.predict(hist) for k, f in fits.items()}}))
xs = pd.concat(rows, ignore_index=True)
print(f"{xs.pair.nunique()} pairs, {len(xs):,} bins")
display(pd.DataFrame({k: scores(xs.y.to_numpy(), xs[k].to_numpy()) for k in CANDIDATES}).T)


def boot_diff(a, b, n=2000, seed=1):
    # mean |error| of a minus b, 95% CI from resampling subjects
    s = (np.abs(xs[a] - xs.y) - np.abs(xs[b] - xs.y)).groupby(xs.ID).agg(["sum", "count"])
    idx = np.random.default_rng(seed).integers(0, len(s), (n, len(s)))
    means = s["sum"].to_numpy()[idx].sum(1) / s["count"].to_numpy()[idx].sum(1)
    return {"Δ MAE L/min": s["sum"].sum() / s["count"].sum(), "CI low": np.quantile(means, 0.025), "CI high": np.quantile(means, 0.975)}


display(pd.DataFrame({f"{a}  vs  {b}": boot_diff(a, b) for a, b in [
    ("C", "A"), ("C, noise-trained", "A"), ("C, noise-trained", "C"), ("B", "A"), ("B, noise-trained", "B"),
    ("D", "B"), ("D, noise-trained", "B, noise-trained")]}).T.round(2))

316 pairs, 26,127 bins


,MAE L/min,median APE %,90th pct APE %,R²
A,8.16,11.24,30.95,0.87
B,7.35,9.85,26.58,0.89
"B, noise-trained",7.29,9.77,26.91,0.89
C,7.97,10.84,29.35,0.88
"C, noise-trained",7.92,10.79,29.33,0.88
D,7.52,10.29,27.06,0.89
"D, noise-trained",7.44,10.15,27.11,0.89


,Δ MAE L/min,CI low,CI high
C vs A,-0.18,-0.59,0.23
"C, noise-trained vs A",-0.24,-0.58,0.10
"C, noise-trained vs C",-0.05,-0.13,0.02
B vs A,-0.80,-1.29,-0.34
"B, noise-trained vs B",-0.07,-0.19,0.06
D vs B,0.16,0.03,0.30
"D, noise-trained vs B, noise-trained",0.15,0.01,0.30


Δ MAE is the first model's mean error minus the second's, so negative means the first is better. The 95% CI comes
from resampling people.

* **Peak speed holds up.** B beats A by 0.80 L/min, about 10%, CI −1.29 to −0.34. That's close to its same-session
  gain.
* **HR at pace helps a little, but it's not proven.** Noise-trained C beats A by 0.24 L/min, about 3%, but the CI
  runs from −0.58 to +0.10, so 116 people aren't enough to settle it. The same-session result in §4 points the same
  way, so C is what runs when HR is all the user has.
* **HR on top of peak speed hurts.** D is 0.16 L/min worse than B, CI +0.03 to +0.30, and noise training doesn't
  fix it. History HR misses the same-day value by about 7 bpm (robust SD). Once peak speed carries the fitness
  signal, HR adds that noise and little else. The final model ignores HR whenever a best effort is given.
* **Noise training is a wash on lab data.** It moves B by −0.07 and C by −0.05 L/min, and both CIs span zero. I keep
  it because race-derived peak speed and wrist HR from outdoor runs will be noisier than one lab test standing in for
  another, and noise training makes the model lean on them less.

## 6. Steady-state speed and HR

The ramp adds 1 km/h a minute, so VE at a given ramp speed is lower than it would be at that speed held steady.
`explore.ipynb` §11 fitted the lag at the start of walking and turned it into a speed shift: evaluate the formula at
goal speed + shift.

HR from a watch is a steady-state reading too, so it needs the same shift as VE. At walking onset HR jumps within
seconds, which says nothing about running. After the peak, though, HR and VE come down at similar rates, which
suggests similar lags on the ramp. So the formula takes the user's HR as is, at the shifted speed.

In [7]:
mono = lambda t, base, amp, tau: base + amp * (1 - np.exp(-t / tau))
decay = lambda t, base, amp, tau: base + amp * np.exp(-t / tau)
dur = m.groupby(["ID_test", "phase"]).time.agg(lambda t: t.max() - t.min()).unstack()

wu = m[m.ID_test.isin(dur.index[dur.warmup >= 150]) & (m.phase == "warmup") & (m.t_phase <= 150)]
wu_curve = wu.groupby((wu.t_phase // 5) * 5 + 2.5).VE.median()
tau_on = curve_fit(mono, wu_curve.index, wu_curve.values, p0=[10, 10, 30])[0][2]
ramp_rate = m[m.phase == "ramp"].groupby("ID_test").apply(
    lambda t: np.polyfit(t.time, t.Speed, 1)[0] * 60 if len(t) > 10 else np.nan, include_groups=False).median()
RAMP_LAG_KMH = ramp_rate * tau_on / 60

rec = m[m.ID_test.isin(dur.index[dur.recovery >= 150]) & (m.phase == "recovery") & m.Speed.between(4.85, 5.05)
        & (m.t_phase <= 180)]
tau_off = {}
for col in ["VE", "HR"]:
    c = rec.groupby((rec.t_phase // 5) * 5 + 2.5)[col].median()
    tau_off[col] = curve_fit(decay, c.index, c.values, p0=[c.iloc[-1], c.iloc[0] - c.iloc[-1], 60])[0][2]
print(f"VE on-kinetics τ {tau_on:.0f} s, ramp {ramp_rate:.2f} km/h per min → evaluate at goal speed + {RAMP_LAG_KMH:.2f} km/h")
print(f"recovery τ: VE {tau_off['VE']:.0f} s, HR {tau_off['HR']:.0f} s")

VE on-kinetics τ 33 s, ramp 1.00 km/h per min → evaluate at goal speed + 0.55 km/h
recovery τ: VE 100 s, HR 116 s


## 7. Error band

The band comes from per-test error, not per-bin error, since a session is one person at one pace. For each test,
take the median of actual ÷ predicted across its bins. The 10th and 90th percentiles of that across tests become
the band.

History error comes in two ways, and neither is perfect:

* real pairs: the §5 predictions, where history really is another test. Honest, but only 116 people, all of whom
  came back for another test.
* resampled: every one of the 992 tests gets an HR and peak-speed offset drawn from the §5 pairs. This covers
  everyone. It overstates peak-speed error, though, because the offset stacks a second test's noise on top of the
  target test's own.

The final band takes the wider of the two at each end.

In [8]:
FINAL = {"A": CANDIDATES["A"], "B": CANDIDATES["B, noise-trained"], "C": CANDIDATES["C, noise-trained"]}
rng = np.random.default_rng(0)
ids = d.ID_test.unique()
draw = hist_err.iloc[rng.integers(0, len(hist_err), len(ids))].set_axis(ids)
d_hist = d.assign(HR=d.HR + d.ID_test.map(draw.hr_off), vpeak=d.vpeak + d.ID_test.map(draw.vpeak_off))

oof_hist = {k: np.empty(len(d)) for k in FINAL}
for tr, te in folds:
    for k, v in FINAL.items():
        oof_hist[k][te] = Formula(d.iloc[tr], **v).predict(d_hist.iloc[te])
print("resampled history error, all tests")
display(pd.DataFrame({k: scores(y, p) for k, p in oof_hist.items()}).T)

q = [0.1, 0.5, 0.9]
resampled = pd.DataFrame({k: pd.Series(y / p).groupby(d.ID_test.values).median().quantile(q) for k, p in oof_hist.items()}).T
pairs_band = pd.DataFrame({k: (xs.y / xs[c]).groupby(xs.pair).median().quantile(q)
                           for k, c in [("A", "A"), ("B", "B, noise-trained"), ("C", "C, noise-trained")]}).T
BAND = pd.DataFrame({"p10": np.minimum(resampled[0.1], pairs_band[0.1]), "p90": np.maximum(resampled[0.9], pairs_band[0.9])})
print("per-session actual ÷ predicted")
display(pd.concat({"resampled": resampled, "real pairs": pairs_band}, axis=1)
        .set_axis([f"{s} {c}" for s in ["resampled", "pairs"] for c in ["p10", "p50", "p90"]], axis=1)
        .join(BAND.add_prefix("band ")).round(2))

resampled history error, all tests


,MAE L/min,median APE %,90th pct APE %,R²
A,8.68,11.81,31.01,0.85
B,8.27,11.10,29.80,0.86
C,8.32,11.10,29.59,0.86


per-session actual ÷ predicted


,resampled p10,resampled p50,resampled p90,pairs p10,pairs p50,pairs p90,band p10,band p90
A,0.83,0.98,1.19,0.80,0.95,1.14,0.80,1.19
B,0.83,0.98,1.16,0.84,0.98,1.13,0.83,1.16
C,0.83,0.98,1.17,0.83,0.97,1.17,0.83,1.17


The 80% band is roughly −17% to +17% around the estimate, and a little wider for A. History narrows it only
slightly. With resampled error, B and C cut MAE by about 5% against A. That's less than B's 10% on real pairs, partly
because resampling double counts peak-speed noise.

None of this covers the error from turning a race time into peak speed (§9). For B, that conversion is the bigger
risk.

## 8. Final model

Refit on every test. These coefficients are the whole model: log VE = intercept + Σ coefficient × term, and VE =
exp(that) × smear.

In [9]:
MODELS = {k: Formula(d, **v) for k, v in FINAL.items()}
terms = ["intercept", *design(d.head(1), hr=True, vpeak=True).columns, "smear"]
display(pd.DataFrame({k: f.coefs() for k, f in MODELS.items()}).reindex(terms)
        .map(lambda c: "" if pd.isna(c) else f"{c:.4g}"))

RANGE = {"speed km/h": (d.Speed.quantile(0.01), d.Speed.quantile(0.99)), "age": (d.Age.min(), d.Age.max()),
         "weight kg": (d.Weight.min(), d.Weight.max()), "height cm": (d.Height.min(), d.Height.max()),
         "peak speed km/h": (vpeak_of.min(), vpeak_of.max()), "HR": (d.HR.quantile(0.01), d.HR.quantile(0.99))}
print("valid range:", ", ".join(f"{k} {lo:.4g}-{hi:.4g}" for k, (lo, hi) in RANGE.items()))

,A,B,C
intercept,0.2765,0.6517,-0.2329
Speed,0.1214,0.1617,0.125
"max(0,Speed-6.5)",0.1448,0.1405,0.1453
"max(0,Speed-8)",-0.1331,-0.133,-0.1027
"max(0,Speed-10)",-0.04725,-0.03936,-0.02555
"max(0,Speed-13)",0.009906,0.01354,0.02873
"max(0,Speed-16)",-0.03248,-0.00276,-0.0172
Age,0.002474,0.001738,0.004411
Sex,0.01213,-0.07326,-0.03132
logW,0.4755,0.338,0.3919


valid range: speed km/h 4.9-18.9, age 10.8-63, weight kg 41-135, height cm 150-203, peak speed km/h 7-23.1, HR 73-198


## 9. From running history to model inputs

**Peak speed from a best effort.** Riegel's formula, T₂ = T₁ × (D₂/D₁)^1.06, turns the effort into a 5 km
equivalent. Then vpeak = 5 km speed ÷ 0.876. The 0.876 comes from Figueiredo et al. 2021, where recreational runners
did 5 km at 12.0 km/h against a Vpeak of 13.7 km/h. Their test used 3-minute stages, and this dataset's ramp adds
1 km/h every minute. Faster ramps give higher peaks, so 0.876 probably understates vpeak here, and B leans high.
That's the cautious direction for an exposure warning, but it's the least certain number in the notebook.

**HR at the goal pace.** This fits a straight line of average HR against speed across the recent runs, and reads
it at the goal speed. It's only used within 0.5 km/h of the runs' speed range. With one run, it's used if that run
was within 0.5 km/h of the goal speed.

In [10]:
RIEGEL = 1.06
FIVE_K_OVER_VPEAK = 0.876


def to_minutes(s):
    parts = [float(p) for p in str(s).split(":")]
    return sum(p * 60 ** (len(parts) - 2 - i) for i, p in enumerate(parts))


def vpeak_from_best_effort(be):
    t5 = to_minutes(be["time"]) * (5 / be["distance_km"]) ** RIEGEL
    return 5 / (t5 / 60) / FIVE_K_OVER_VPEAK


def hr_at_speed(runs, speed, tol=0.5):
    if not runs:
        return None, "no runs with HR"
    sp = np.array([r["distance_km"] / (to_minutes(r["time"]) / 60) for r in runs])
    hr = np.array([r["avg_hr"] for r in runs], dtype=float)
    if not sp.min() - tol <= speed <= sp.max() + tol:
        return None, f"goal speed {speed:.1f} km/h outside run history ({sp.min():.1f}-{sp.max():.1f} km/h)"
    if len(sp) == 1 or np.ptp(sp) < 0.3:
        return float(hr.mean()), f"mean HR of runs at {sp.mean():.1f} km/h"
    slope, icpt = np.polyfit(sp, hr, 1)
    return float(icpt + slope * speed), f"HR-speed line over {len(sp)} runs, {slope:.1f} bpm per km/h"

## 10. Estimate

The run is assumed to hold a steady pace on flat ground. The data can't speak to heat, humidity, hills, or VE
drifting upward during runs much longer than the ~11 minute lab ramps, so the warnings say when the inputs fall
outside it.

In [11]:
def features(plan, person, hr=np.nan, vpeak=np.nan):
    # one model row; running speeds shifted to the ramp speed whose VE matches steady state (§6)
    speed = 60 / to_minutes(plan["pace"])
    return pd.DataFrame({"Speed": [speed + RAMP_LAG_KMH if speed > 5.5 else speed], "Age": person["age"],
                         "Sex": int(person["sex"] == "female"), "Weight": person["weight_kg"],
                         "Height": person["height_cm"], "HR": hr, "vpeak": vpeak})


def estimate_ve(plan, person, history=None):
    history = history or {}
    pace = to_minutes(plan["pace"])
    speed = 60 / pace
    minutes = plan["distance_km"] * pace
    x = features(plan, person)
    notes, warn = [], []

    be = history.get("best_effort")
    if be:
        x["vpeak"] = vpeak_from_best_effort(be)
        notes.append(f"peak speed {x.vpeak[0]:.1f} km/h from best effort {be['distance_km']:g} km in {be['time']} "
                     f"(lab median {vpeak_of.median():.1f})")
        goal_time = to_minutes(be["time"]) * (plan["distance_km"] / be["distance_km"]) ** RIEGEL
        if minutes < goal_time:
            mm, ss = divmod(round(goal_time / plan["distance_km"] * 60), 60)
            warn.append(f"goal pace is faster than the best effort predicts for {plan['distance_km']:g} km "
                        f"({mm}:{ss:02d} min/km); VE will be higher than estimated")
    hr, how = hr_at_speed(history.get("runs") or [], speed)
    if hr is not None:
        x["HR"] = hr
        lab_hr = d.HR[d.Speed.between(x.Speed[0] - 0.25, x.Speed[0] + 0.25)].median()
        notes.append(f"HR at pace {hr:.0f} bpm ({how}; lab median at this pace {lab_hr:.0f})")
    elif history.get("runs"):
        warn.append(f"HR not used: {how}")

    variant = "B" if be else "C" if hr is not None else "A"
    ve = MODELS[variant].predict(x)[0]
    band = BAND.loc[variant]

    checks = {"speed km/h": x.Speed[0], "age": person["age"], "weight kg": person["weight_kg"],
              "height cm": person["height_cm"], "peak speed km/h": x.vpeak[0], "HR": x.HR[0]}
    for k, v in checks.items():
        lo, hi = RANGE[k]
        if not np.isnan(v) and not lo <= v <= hi:
            warn.append(f"{k} {v:.4g} outside the data ({lo:.4g}-{hi:.4g})")
    return {"variant": variant, "speed_kmh": speed, "minutes": minutes, "ve_lmin": ve,
            "ve_p10": ve * band.p10, "ve_p90": ve * band.p90, "air_m3": ve * minutes / 1000,
            "air_p10_m3": ve * band.p10 * minutes / 1000, "air_p90_m3": ve * band.p90 * minutes / 1000,
            "notes": notes, "warnings": warn}


r = estimate_ve(PLAN, PERSON, HISTORY)
print(f"Plan: {PLAN['distance_km']:g} km at {PLAN['pace']} min/km ({r['speed_kmh']:.1f} km/h, {r['minutes']:.0f} min)")
print(f"Model {r['variant']}" + "".join(f"\n  · {n}" for n in r["notes"]))
print(f"\nExpected VE      {r['ve_lmin']:.1f} L/min   (80% band {r['ve_p10']:.1f}-{r['ve_p90']:.1f})")
print(f"Air over the run {r['air_m3']:.2f} m³      (80% band {r['air_p10_m3']:.2f}-{r['air_p90_m3']:.2f})")
for w in r["warnings"]:
    print(f"  ! {w}")

Plan: 10 km at 6:00 min/km (10.0 km/h, 60 min)
Model B
  · peak speed 13.7 km/h from best effort 5 km in 25:00 (lab median 16.7)
  · HR at pace 156 bpm (HR-speed line over 3 runs, 10.5 bpm per km/h; lab median at this pace 154)

Expected VE      72.6 L/min   (80% band 60.3-84.4)
Air over the run 4.35 m³      (80% band 3.62-5.07)


For comparison, the same plan with less history, and how much the HR input moves the C estimate:

In [12]:
cases = {"no history (A)": None,
         "runs with HR only (C)": {"runs": HISTORY.get("runs")},
         "best effort (B)": {"best_effort": HISTORY.get("best_effort"), "runs": HISTORY.get("runs")}}
display(pd.DataFrame({k: estimate_ve(PLAN, PERSON, h) for k, h in cases.items()})
        .loc[["variant", "ve_lmin", "ve_p10", "ve_p90", "air_m3"]].T)

if HISTORY.get("runs"):
    hr0, _ = hr_at_speed(HISTORY["runs"], 60 / to_minutes(PLAN["pace"]))
    if hr0 is not None:
        ve_c = lambda h: MODELS["C"].predict(features(PLAN, PERSON, hr=h))[0]
        print("model C, HR shifted:", ", ".join(f"{dh:+d} bpm → {100 * (ve_c(hr0 + dh) / ve_c(hr0) - 1):+.1f}%"
                                                for dh in [-10, -5, 5, 10]))

,variant,ve_lmin,ve_p10,ve_p90,air_m3
no history (A),A,64.64,51.78,77.11,3.88
runs with HR only (C),C,64.83,53.77,75.72,3.89
best effort (B),B,72.56,60.3,84.44,4.35


model C, HR shifted: -10 bpm → -4.1%, -5 bpm → -2.1%, +5 bpm → +2.3%, +10 bpm → +5.7%


For the example runner, a 30-year-old man of 70 kg running 10 km at 6:00/km:

* **No history (A).** 64.6 L/min, 3.88 m³ over the hour.
* **HR runs only (C).** 64.8 L/min. His 156 bpm at this pace is close to the lab median of 154, so HR barely moves
  it. Each 5 bpm of HR error shifts C by about 2%.
* **Best effort (B).** 72.6 L/min, 4.35 m³. A 25:00 5 km maps to a peak speed of 13.7 km/h, well under the lab
  median of 16.7. B reads him as less fit than the average lab runner and raises VE 12%. If 0.876 understates peak
  speed for this ramp, as §9 suspects, part of that 12% isn't real.

Known gaps, all outside what the lab data covers:

* Heat and humidity. The lab ran at 15 to 32 °C and about 48% RH. Singapore runs are hotter and wetter, and heat
  pushes VE up.
* Long runs. Lab ramps lasted about 11 minutes, and VE drifts upward over an hour, so long sessions likely come out
  low.
* Walking groups and older adults. Speeds under 4.9 km/h and ages over 63 only trigger a warning, and nothing here
  replaces the estimate for them.
* Hills, wind and surface.